# Paper 2 — Goal 3, Stage B
# Signal-only perturbation calibration and dose freeze

This notebook is the **outcome-blind calibration stage** for Goal 3.

It does not read diagnosis, bulbar score, Goal 2 predictions, prediction error, or any Goal 3 clinical-model response.

Its purpose is only to answer:

> **What physical low / medium / high acquisition perturbation settings produce ordered, technically valid changes in the prespecified Q measurements?**

The scientific order is:

1. load the already-frozen Goal 3 Stage-A signal-only pilot;
2. verify raw media identity and the pinned Paper 1 signal-processing implementation;
3. rerun canonical waveform preprocessing and Silero segmentation;
4. verify that the exact Q extractor reproduces baseline Q on the pilot;
5. apply candidate physical perturbations;
6. rerun preprocessing + segmentation + Q extraction;
7. quantify target-Q movement and off-target Q movement;
8. choose low / medium / high doses **using Q response only**;
9. freeze `goal3_perturbation_manifest.csv`;
10. only after this manifest is frozen may the controlled clinical inference stage inspect Goal 2 model responses.

### Critical interpretation boundary

Dose selection is an engineering calibration problem, not a clinical-model optimization problem.

No downstream clinical result is permitted to alter the frozen dose grid.


In [1]:
from __future__ import annotations

import hashlib
import importlib
import inspect
import json
import math
import os
import platform
import shutil
import subprocess
import sys
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from scipy import signal

ENGINE_VERSION = "goal3-dose-calibration-v1.0.0"
PAPER1_COMMIT = "cb31fb6886df1b2b2fedba4ffbbf8624bd56d7e8"
BASE_SEED = 20260825

EXPECTED_PILOT_N = 24
TARGET_IQR_MULTIPLES = {
    "low": 0.5,
    "medium": 1.0,
    "high": 2.0,
}

# Automatic freezing is deliberately guarded. Leave True:
# the notebook freezes only if every family passes all signal-only gates.
AUTO_FREEZE_IF_ALL_PASS = True

def find_project_root() -> Path:
    override = os.environ.get("PAPER2_ROOT", "").strip()
    if override:
        root = Path(override).expanduser().resolve()
        if (root / "data" / "processed").exists():
            return root
        raise FileNotFoundError(f"Invalid PAPER2_ROOT: {root}")

    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (
            (candidate / "data" / "processed" / "recording_table_phase0.csv").exists()
            and (candidate / "outputs" / "goal3").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate Paper 2 root. Run from the repository or notebooks folder."
    )

ROOT = find_project_root()
MANIFESTS = ROOT / "data" / "manifests"
PROCESSED = ROOT / "data" / "processed"
OUT = ROOT / "outputs" / "goal3" / "stageB_signal_only_calibration_v1_0"
TABLES = OUT / "tables"
AUDIT = OUT / "audit"
FIGURES = OUT / "figures"
CACHE = OUT / "cache"

for directory in [OUT, TABLES, AUDIT, FIGURES, CACHE]:
    directory.mkdir(parents=True, exist_ok=True)

print("Paper 2 root:", ROOT)
print("Engine:", ENGINE_VERSION)
print("Paper 1 pinned commit:", PAPER1_COMMIT)


Paper 2 root: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code
Engine: goal3-dose-calibration-v1.0.0
Paper 1 pinned commit: cb31fb6886df1b2b2fedba4ffbbf8624bd56d7e8


## 1. Atomic I/O and deterministic utilities

In [2]:
def sha256_file(path: Path, chunk_size=1024 * 1024) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def stable_seed(*parts) -> int:
    payload = "|".join(map(str, parts))
    digest = hashlib.sha256(payload.encode("utf-8")).hexdigest()
    return (BASE_SEED + int(digest[:8], 16)) % (2**32 - 1)

def atomic_csv(frame: pd.DataFrame, path: Path, *, allow_empty=False):
    path.parent.mkdir(parents=True, exist_ok=True)
    if not allow_empty and len(frame) == 0:
        raise ValueError(f"Refusing to write empty table: {path}")
    temp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temp, index=False)
    if temp.stat().st_size == 0:
        raise IOError(f"Temporary output is empty: {temp}")
    os.replace(temp, path)

def atomic_json(payload: dict, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_suffix(path.suffix + ".tmp")
    temp.write_text(
        json.dumps(payload, indent=2, default=str),
        encoding="utf-8",
    )
    os.replace(temp, path)

def read_csv_checked(path: Path, required=None) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)
    if path.stat().st_size == 0:
        raise RuntimeError(f"Empty CSV: {path}")
    frame = pd.read_csv(path, low_memory=False)
    if required:
        missing = set(required) - set(frame.columns)
        if missing:
            raise KeyError(f"{path.name}: missing {sorted(missing)}")
    return frame

print("UTILITY GATE: PASS")


UTILITY GATE: PASS


## 2. Locate and validate Goal 3 Stage-A outputs

The calibration notebook searches both `stageA_v1_1` and the older `stageA_v1_0` output directory, because the earlier v1.1 logic was initially written into the v1.0 folder.

Only Stage-A signal-only artifacts are loaded.


In [3]:
stage_a_candidates = [
    ROOT / "outputs" / "goal3" / "stageA_v1_1",
    ROOT / "outputs" / "goal3" / "stageA_v1_0",
]

STAGE_A = next(
    (
        path
        for path in stage_a_candidates
        if (path / "SUCCESS_STAGE_A.json").exists()
    ),
    None,
)

if STAGE_A is None:
    raise FileNotFoundError(
        "No successful Goal 3 Stage-A directory found."
    )

stage_a_success = json.loads(
    (STAGE_A / "SUCCESS_STAGE_A.json").read_text(encoding="utf-8")
)

if stage_a_success.get("status") != "PASS":
    raise RuntimeError("Goal 3 Stage A is not PASS.")

STAGE_A_TABLES = STAGE_A / "tables"

pilot = read_csv_checked(
    STAGE_A_TABLES / "goal3_signal_only_dose_pilot.csv",
    required=[
        "participant_id",
        "logical_recording_id",
        "pilot_rank",
    ],
)

natural_iqr = read_csv_checked(
    STAGE_A_TABLES / "goal3_natural_q_iqr_targets.csv",
    required=[
        "q_feature",
        "natural_iqr_raw",
    ],
)

design = read_csv_checked(
    STAGE_A_TABLES / "goal3_perturbation_design_skeleton.csv",
    required=[
        "family",
        "transform",
        "primary_target_q",
        "dose_status",
    ],
)

crossfit = read_csv_checked(
    STAGE_A_TABLES / "goal3_fixed_fivefold_crossfit.csv",
    required=[
        "participant_id",
        "outer_fold",
    ],
)

if len(pilot) != EXPECTED_PILOT_N:
    raise RuntimeError(
        f"Expected {EXPECTED_PILOT_N} pilot sources; found {len(pilot)}."
    )

# Outcome-blinding gate.
PROHIBITED = {
    "diagnosis",
    "y_dx",
    "bulbar_score",
    "assessment_date",
    "age_at_recording_years",
    "prediction",
    "loss",
    "absolute_error",
    "brier",
}

leaked = PROHIBITED & set(pilot.columns)
if leaked:
    raise RuntimeError(
        "Clinical information leaked into signal-only calibration pilot: "
        + ", ".join(sorted(leaked))
    )

pilot["participant_id"] = pilot["participant_id"].astype(str)
pilot["logical_recording_id"] = pilot["logical_recording_id"].astype(str)

pilot = pilot.merge(
    crossfit,
    on="participant_id",
    how="left",
    validate="one_to_one",
)

if pilot["outer_fold"].isna().any():
    raise RuntimeError("At least one pilot participant lacks a fixed Goal 3 fold.")

print("STAGE-A INPUT GATE: PASS")
print("Stage-A directory:", STAGE_A)
display(design)


STAGE-A INPUT GATE: PASS
Stage-A directory: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\outputs\goal3\stageA_v1_0


,family,transform,exemplars_per_nonzero_dose,physical_dose_variable,primary_target_q,secondary_target_q,expected_primary_direction,dose_status
0,QADD,stationary_colored_broadband,5,injected_SNR_dB_relative_to_baseline_strict_sp...,qadd_pause_ac_level_dbfs_median,qadd_speech_pause_level_contrast_db,increase_pause_level,UNFROZEN_SIGNAL_ONLY_CALIBRATION_REQUIRED
1,QADD,amplitude_modulated_colored_broadband,5,injected_SNR_dB_relative_to_baseline_strict_sp...,qadd_pause_level_iqr_db,qadd_pause_ac_level_dbfs_median | qadd_speech_...,increase_pause_level_variability,UNFROZEN_SIGNAL_ONLY_CALIBRATION_REQUIRED
2,QGAIN,uniform_level_shift,1,gain_dB,qgain_typical_speech_level_dbfs,NaN,known_signed_shift,UNFROZEN_SIGNAL_ONLY_CALIBRATION_REQUIRED
3,QGAIN,smooth_time_varying_gain,1,gain_curve_amplitude_and_shape,qgain_within_segment_iqr_db,qgain_between_segment_mad_db | qgain_abs_drift...,increase_level_dynamics,UNFROZEN_SIGNAL_ONLY_CALIBRATION_REQUIRED
4,QREV,RIR_convolution_RMS_matched,3,prespecified_RIR_exemplar_and_smearing_level,qrev_srmr_norm,NaN,decrease_normalized_fast_SRMR,UNFROZEN_SIGNAL_ONLY_CALIBRATION_REQUIRED
5,QCHAN,upper_band_restriction,3,filter_exemplar_and_cutoff_or_shape,qchan_rolloff95_deficit_hz,qchan_highband_ratio_deficit,increase_upper_band_deficits,UNFROZEN_SIGNAL_ONLY_CALIBRATION_REQUIRED
6,QDIST,symmetric_hard_clipping,1,validated_Paper1_hard_clip_burden,qdist_hard_clipped_sample_fraction,NaN,increase_hard_clipped_sample_fraction,UNFROZEN_USE_VALIDATED_PAPER1_CHALLENGE_GRID


## 3. Resolve the pinned Paper 1 implementation

No Q algorithm is reimplemented here.

The notebook imports the exact reviewed Paper 1 modules from commit
`cb31fb6886df1b2b2fedba4ffbbf8624bd56d7e8`.

If a local checkout is not found, the notebook clones a read-only pinned copy into `vendor/quality_framework_features`.


In [4]:
def git_output(args, cwd=None):
    result = subprocess.run(
        ["git", *args],
        cwd=cwd,
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"git {' '.join(args)} failed:\n{result.stderr}"
        )
    return result.stdout.strip()

repo_candidates = [
    ROOT / "vendor" / "quality_framework_features",
    ROOT.parent / "quality_framework_features",
    Path.home() / "Desktop" / "quality_framework_features",
]

PAPER1_REPO = next(
    (path.resolve() for path in repo_candidates if (path / ".git").exists()),
    None,
)

if PAPER1_REPO is None:
    PAPER1_REPO = (ROOT / "vendor" / "quality_framework_features").resolve()
    PAPER1_REPO.parent.mkdir(parents=True, exist_ok=True)

    print("Pinned Paper 1 checkout not found locally; cloning repository...")
    git_output([
        "clone",
        "https://github.com/nevena-m3/quality_framework_features.git",
        str(PAPER1_REPO),
    ])

observed_commit = git_output(
    ["rev-parse", "HEAD"],
    cwd=PAPER1_REPO,
)

if observed_commit != PAPER1_COMMIT:
    # Do not modify another working checkout with uncommitted changes.
    status = git_output(
        ["status", "--porcelain"],
        cwd=PAPER1_REPO,
    )

    if status:
        raise RuntimeError(
            "Existing Paper 1 checkout is not at the pinned commit and has "
            "uncommitted changes. Use a clean checkout under vendor/."
        )

    git_output(
        ["fetch", "--all", "--tags"],
        cwd=PAPER1_REPO,
    )
    git_output(
        ["checkout", "--detach", PAPER1_COMMIT],
        cwd=PAPER1_REPO,
    )

observed_commit = git_output(
    ["rev-parse", "HEAD"],
    cwd=PAPER1_REPO,
)

if observed_commit != PAPER1_COMMIT:
    raise RuntimeError("Could not pin exact Paper 1 commit.")

src_path = PAPER1_REPO / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

print("PAPER 1 CODE GATE: PASS")
print("Repository:", PAPER1_REPO)
print("Commit:", observed_commit)


Pinned Paper 1 checkout not found locally; cloning repository...
PAPER 1 CODE GATE: PASS
Repository: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\vendor\quality_framework_features
Commit: cb31fb6886df1b2b2fedba4ffbbf8624bd56d7e8


## 4. Dependency gate

Controlled calibration uses the same Silero segmentation algorithm as Paper 1. The notebook therefore refuses to freeze doses if that runtime is not available.

This is intentional: we do not substitute a different VAD or use an ad-hoc segmentation rule.


In [5]:
dependency_rows = []

def check_import(name):
    try:
        module = importlib.import_module(name)
        version = getattr(module, "__version__", "installed")
        dependency_rows.append({
            "dependency": name,
            "available": True,
            "version": str(version),
            "error": "",
        })
        return module
    except Exception as exc:
        dependency_rows.append({
            "dependency": name,
            "available": False,
            "version": "",
            "error": repr(exc),
        })
        return None

torch_mod = check_import("torch")
silero_mod = check_import("silero_vad")
gammatone_mod = check_import("gammatone")

for executable in ["ffmpeg", "ffprobe"]:
    path = shutil.which(executable)
    dependency_rows.append({
        "dependency": executable,
        "available": path is not None,
        "version": path or "",
        "error": "",
    })

dependency_audit = pd.DataFrame(dependency_rows)
atomic_csv(
    dependency_audit,
    AUDIT / "dependency_audit.csv",
)

display(dependency_audit)

missing = dependency_audit.loc[
    ~dependency_audit["available"],
    "dependency",
].tolist()

if missing:
    print("\nDEPENDENCY GATE: NOT READY")
    print("Missing:", missing)
    print(
        "\nDo not substitute another segmentation/reverberation implementation. "
        "The calibration cannot be frozen until these pinned-runtime dependencies "
        "are available."
    )
else:
    print("DEPENDENCY GATE: PASS")


,dependency,available,version,error
0,torch,False,,"ModuleNotFoundError(""No module named 'torch'"")"
1,silero_vad,False,,"ModuleNotFoundError(""No module named 'silero_v..."
2,gammatone,False,,"ModuleNotFoundError(""No module named 'gammaton..."
3,ffmpeg,True,C:\ffmpeg\bin\ffmpeg.EXE,
4,ffprobe,True,C:\ffmpeg\bin\ffprobe.EXE,



DEPENDENCY GATE: NOT READY
Missing: ['torch', 'silero_vad', 'gammatone']

Do not substitute another segmentation/reverberation implementation. The calibration cannot be frozen until these pinned-runtime dependencies are available.


## 5. Import exact Paper 1 measurement modules and validate APIs

In [6]:
from paper1_qc.media import decode_native_audio
from paper1_qc.segmentation import (
    Interval as SegInterval,
    silero_speech_intervals,
    build_segmentation_views,
    load_silero_model,
)

from paper1_qc_reviewed.qadd_v420 import (
    extract_qadd,
    TimeInterval as QADDInterval,
)

from paper1_qc_reviewed.qgain_v410 import (
    extract_qgain,
    TimeInterval as QGAINInterval,
)

from paper1_qc_reviewed.qrev_v400 import (
    extract_qrev,
    SpeechInterval as QREVInterval,
)

from paper1_qc_reviewed.qchan_v400 import (
    extract_recording_spectrum,
    compute_reference_relative_features,
    build_subject_balanced_loso_references,
    TimeInterval as QCHANInterval,
)

from paper1_qc import qdist_v410_candidate as qdist_detector

required_api = {
    "extract_qadd": extract_qadd,
    "extract_qgain": extract_qgain,
    "extract_qrev": extract_qrev,
    "extract_recording_spectrum": extract_recording_spectrum,
    "compute_reference_relative_features": compute_reference_relative_features,
    "build_subject_balanced_loso_references": build_subject_balanced_loso_references,
    "silero_speech_intervals": silero_speech_intervals,
    "build_segmentation_views": build_segmentation_views,
}

api_audit = pd.DataFrame([
    {
        "function": name,
        "module": fn.__module__,
        "signature": str(inspect.signature(fn)),
    }
    for name, fn in required_api.items()
])

atomic_csv(
    api_audit,
    AUDIT / "paper1_api_audit.csv",
)

display(api_audit)
print("MEASUREMENT API GATE: PASS")


,function,module,signature
0,extract_qadd,paper1_qc_reviewed.qadd_v420,"(waveform: 'np.ndarray', sample_rate: 'int', *..."
1,extract_qgain,paper1_qc_reviewed.qgain_v410,"(waveform: 'np.ndarray', fs: 'int', *, strict_..."
2,extract_qrev,paper1_qc_reviewed.qrev_v400,"(waveform, fs, *, primary_speech, strict_speec..."
3,extract_recording_spectrum,paper1_qc_reviewed.qchan_v400,"(waveform: 'np.ndarray', sample_rate_hz: 'int'..."
4,compute_reference_relative_features,paper1_qc_reviewed.qchan_v400,"(observation: 'RecordingSpectrum', reference: ..."
5,build_subject_balanced_loso_references,paper1_qc_reviewed.qchan_v400,"(spectra: 'Mapping[str, RecordingSpectrum]', m..."
6,silero_speech_intervals,paper1_qc.segmentation,"(waveform_16k: 'np.ndarray', *, threshold: 'fl..."
7,build_segmentation_views,paper1_qc.segmentation,"(raw_speech: 'Iterable[Interval]', *, duration..."


MEASUREMENT API GATE: PASS


## 6. Resolve pilot media and verify source hashes

The Stage-A pilot was generated without clinical labels. We now attach only source-media identity from the canonical Phase-0 recording table.


In [7]:
recording_table = read_csv_checked(
    PROCESSED / "recording_table_phase0.csv",
    required=[
        "participant_id",
        "logical_recording_id",
    ],
)

recording_table["participant_id"] = recording_table["participant_id"].astype(str)
recording_table["logical_recording_id"] = (
    recording_table["logical_recording_id"].astype(str)
)

media_cols = [
    column
    for column in [
        "participant_id",
        "logical_recording_id",
        "media_path",
        "media_sha256",
    ]
    if column in recording_table.columns
]

pilot_media = pilot.merge(
    recording_table[media_cols],
    on=["participant_id", "logical_recording_id"],
    how="left",
    validate="one_to_one",
    suffixes=("", "_canonical"),
)

if "media_path" not in pilot_media.columns:
    raise RuntimeError(
        "Canonical recording table does not provide media_path."
    )

def resolve_media_path(raw):
    path = Path(str(raw))
    if path.exists():
        return path.resolve()

    # Common fallback: source path may have been written relative to repo root.
    candidate = ROOT / path
    if candidate.exists():
        return candidate.resolve()

    return None

pilot_media["resolved_media_path"] = [
    str(path) if path is not None else ""
    for path in pilot_media["media_path"].map(resolve_media_path)
]

pilot_media["media_exists"] = (
    pilot_media["resolved_media_path"].astype(str).str.len() > 0
)

if not pilot_media["media_exists"].all():
    display(
        pilot_media.loc[
            ~pilot_media["media_exists"],
            ["participant_id", "logical_recording_id", "media_path"],
        ]
    )
    raise FileNotFoundError(
        "At least one signal-only pilot media file cannot be resolved."
    )

hash_rows = []
for row in pilot_media.itertuples(index=False):
    path = Path(row.resolved_media_path)
    observed = sha256_file(path)
    expected = getattr(row, "media_sha256", None)

    matches = (
        True
        if expected is None or pd.isna(expected) or str(expected).strip() == ""
        else observed == str(expected)
    )

    hash_rows.append({
        "participant_id": row.participant_id,
        "logical_recording_id": row.logical_recording_id,
        "resolved_media_path": str(path),
        "observed_sha256": observed,
        "expected_sha256": "" if expected is None else str(expected),
        "sha256_matches": matches,
    })

media_audit = pd.DataFrame(hash_rows)
atomic_csv(
    media_audit,
    AUDIT / "pilot_media_hash_audit.csv",
)

if not media_audit["sha256_matches"].all():
    raise RuntimeError(
        "Pilot media SHA-256 mismatch. Calibration stopped."
    )

pilot_media = pilot_media.drop(
    columns=["media_path"],
    errors="ignore",
).merge(
    media_audit[
        ["participant_id", "logical_recording_id", "resolved_media_path"]
    ],
    on=["participant_id", "logical_recording_id"],
    how="left",
    validate="one_to_one",
)

print("PILOT MEDIA IDENTITY GATE: PASS")


PILOT MEDIA IDENTITY GATE: PASS


## 7. Canonical re-segmentation adapter

Every candidate waveform is re-segmented from scratch with the Paper 1 Silero configuration.

No baseline segmentation is reused in the primary calibration path.


In [8]:
ANALYSIS_SR = 16_000

def native_to_analysis_16k(native: np.ndarray, source_sr: int) -> np.ndarray:
    native = np.asarray(native, dtype=np.float32)

    if native.ndim == 1:
        mono = native.astype(np.float32, copy=False)
    elif native.ndim == 2:
        mono = native.mean(axis=1, dtype=np.float64).astype(np.float32)
    else:
        raise ValueError("native waveform must be samples or samples x channels")

    mono_dc = mono - float(np.mean(mono, dtype=np.float64))

    if source_sr == ANALYSIS_SR:
        analysis = mono_dc.astype(np.float32, copy=True)
    else:
        divisor = math.gcd(int(source_sr), ANALYSIS_SR)
        analysis = signal.resample_poly(
            mono_dc,
            up=ANALYSIS_SR // divisor,
            down=int(source_sr) // divisor,
        ).astype(np.float32)

    if not np.isfinite(analysis).all():
        raise RuntimeError("Analysis waveform contains NaN/Inf.")

    return analysis

SILERO_MODEL = None

def canonical_segmentation(analysis_16k: np.ndarray):
    global SILERO_MODEL

    if SILERO_MODEL is None:
        SILERO_MODEL = load_silero_model(onnx=True)

    raw = silero_speech_intervals(
        analysis_16k,
        threshold=0.5,
        min_speech_ms=250,
        min_silence_ms=100,
        speech_pad_ms=0,
        onnx=True,
        model=SILERO_MODEL,
    )

    views = build_segmentation_views(
        raw,
        duration_sec=len(analysis_16k) / ANALYSIS_SR,
        bridge_gap_ms=100,
        min_speech_ms=250,
        strict_speech_edge_ms=50,
        strict_nonspeech_edge_ms=200,
    )

    return views

def convert_intervals(intervals, cls, *, view=None):
    converted = []

    for idx, item in enumerate(intervals):
        if cls is QREVInterval:
            converted.append(
                cls(
                    float(item.start_sec),
                    float(item.end_sec),
                    f"{view or 'segment'}_{idx}",
                    idx,
                    view or "primary_speech",
                    "primary",
                )
            )
        else:
            converted.append(
                cls(
                    float(item.start_sec),
                    float(item.end_sec),
                )
            )

    return converted

print("CANONICAL SEGMENTATION ADAPTER: READY")


CANONICAL SEGMENTATION ADAPTER: READY


## 8. Candidate perturbation engine

The candidate grids are intentionally broad. They are **not** the final dose grid.

Low / medium / high will be chosen automatically from the achieved target-Q response, not by looking at any clinical outcome.


In [9]:
# ------------------------------------------------------------------
# QADD: 1/f PSD ("pink") synthetic broadband interference
# ------------------------------------------------------------------

QADD_SNR_GRID_DB = [35, 30, 25, 20, 15, 10, 5, 0]
QADD_REALIZATIONS = 5

# ------------------------------------------------------------------
# QGAIN static: attenuation only avoids clipping as a competing mechanism
# ------------------------------------------------------------------

QGAIN_STATIC_DB = [-0.5, -1, -2, -3, -4, -6, -8, -10]

# ------------------------------------------------------------------
# QGAIN dynamic: zero-mean smooth dB modulation
# ------------------------------------------------------------------

QGAIN_DYNAMIC_AMPLITUDE_DB = [0.5, 1, 1.5, 2, 3, 4, 6, 8]
QGAIN_DYNAMIC_HZ = 0.10

# ------------------------------------------------------------------
# QREV: deterministic synthetic RIRs, 3 exemplars per RT60 candidate
# The final transform manifest records every seed and RT60.
# ------------------------------------------------------------------

QREV_RT60_SEC = [0.15, 0.25, 0.40, 0.60, 0.90, 1.20]
QREV_EXEMPLARS = 3

# ------------------------------------------------------------------
# QCHAN: low-pass restriction candidates, 3 filter transition exemplars
# ------------------------------------------------------------------

QCHAN_CUTOFF_HZ = [7000, 6500, 6000, 5500, 5000, 4500, 4000, 3500]
QCHAN_EXEMPLARS = 3

# QDIST is intentionally not inferred from natural Q distribution here.
# A later cell attempts to recover the validated Paper 1 known-truth challenge
# target fractions from the pinned code/output contract.

def rms(values):
    values = np.asarray(values, dtype=np.float64)
    return float(np.sqrt(np.mean(values * values))) if len(values) else np.nan

def colored_broadband_noise(n, sr, seed):
    rng = np.random.default_rng(seed)

    # Frequency-domain 1/f PSD -> 1/sqrt(f) amplitude.
    frequencies = np.fft.rfftfreq(n, d=1.0 / sr)
    spectrum = (
        rng.normal(size=len(frequencies))
        + 1j * rng.normal(size=len(frequencies))
    )

    amplitude = np.ones_like(frequencies)
    positive = frequencies > 0
    amplitude[positive] = 1.0 / np.sqrt(frequencies[positive])
    amplitude[~positive] = 0.0

    # Keep the intervention broadband but suppress sub-audible/DC energy.
    amplitude[frequencies < 60] = 0.0

    shaped = np.fft.irfft(
        spectrum * amplitude,
        n=n,
    ).astype(np.float64)

    shaped -= np.mean(shaped)
    value_rms = rms(shaped)

    if not np.isfinite(value_rms) or value_rms <= 0:
        raise RuntimeError("Failed to synthesize broadband interference.")

    return shaped / value_rms

def speech_rms_from_intervals(
    analysis,
    intervals,
    sr=ANALYSIS_SR,
):
    pieces = []
    for item in intervals:
        left = max(0, round(float(item.start_sec) * sr))
        right = min(len(analysis), round(float(item.end_sec) * sr))
        if right > left:
            pieces.append(analysis[left:right])

    if not pieces:
        return rms(analysis)

    return rms(np.concatenate(pieces))

def add_noise_at_snr(
    native,
    source_sr,
    *,
    snr_db,
    seed,
    amplitude_modulated=False,
):
    native = np.asarray(native, dtype=np.float64)

    if native.ndim == 1:
        native2d = native[:, None]
    else:
        native2d = native.copy()

    analysis = native_to_analysis_16k(native2d, source_sr)
    baseline_views = canonical_segmentation(analysis)

    strict = baseline_views["strict_speech"]
    speech_level = speech_rms_from_intervals(
        analysis,
        strict,
        ANALYSIS_SR,
    )

    if not np.isfinite(speech_level) or speech_level <= 0:
        raise RuntimeError("Cannot define injected SNR: no finite speech RMS.")

    target_noise_rms = (
        speech_level
        / (10.0 ** (float(snr_db) / 20.0))
    )

    # Generate at native rate, same synthetic field into each decoded channel.
    noise = colored_broadband_noise(
        native2d.shape[0],
        source_sr,
        seed,
    )

    if amplitude_modulated:
        time = np.arange(len(noise)) / float(source_sr)
        # Strictly positive, smooth, deterministic envelope.
        envelope = 0.35 + 0.65 * (
            0.5 + 0.5 * np.sin(
                2 * np.pi * 0.35 * time
                + 0.37
            )
        )
        noise = noise * envelope
        noise = noise / rms(noise)

    noise = noise * target_noise_rms

    out = native2d + noise[:, None]

    return out.astype(np.float32)

def apply_static_gain(native, gain_db):
    factor = 10.0 ** (float(gain_db) / 20.0)
    return (
        np.asarray(native, dtype=np.float64) * factor
    ).astype(np.float32)

def apply_dynamic_gain(
    native,
    source_sr,
    amplitude_db,
    frequency_hz=QGAIN_DYNAMIC_HZ,
):
    values = np.asarray(native, dtype=np.float64)

    if values.ndim == 1:
        values = values[:, None]

    time = np.arange(values.shape[0]) / float(source_sr)

    gain_db = (
        float(amplitude_db)
        * np.sin(
            2.0 * np.pi * float(frequency_hz) * time
        )
    )

    gain = 10.0 ** (gain_db / 20.0)

    return (
        values * gain[:, None]
    ).astype(np.float32)

def synthetic_rir(
    sr,
    rt60_sec,
    exemplar,
):
    # Deterministic exponentially decaying diffuse tail + direct impulse.
    seed = stable_seed(
        "QREV",
        rt60_sec,
        exemplar,
    )
    rng = np.random.default_rng(seed)

    duration = max(0.35, min(2.0, float(rt60_sec) * 1.35))
    n = int(round(duration * sr))
    time = np.arange(n) / float(sr)

    tau = float(rt60_sec) / math.log(1000.0)
    decay = np.exp(-time / max(tau, 1e-4))

    tail = rng.normal(size=n) * decay
    tail[0] = 0.0

    # Direct path + modest diffuse tail.
    rir = 0.78 * np.eye(1, n, 0).ravel() + 0.22 * tail

    # Normalize energy, not peak.
    e = np.sqrt(np.sum(rir * rir))
    if e <= 0 or not np.isfinite(e):
        raise RuntimeError("Invalid synthetic RIR.")

    return rir / e

def apply_rir(native, source_sr, rt60_sec, exemplar):
    values = np.asarray(native, dtype=np.float64)

    if values.ndim == 1:
        values = values[:, None]

    rir = synthetic_rir(
        source_sr,
        rt60_sec,
        exemplar,
    )

    before = rms(values.reshape(-1))

    channels = []
    for c in range(values.shape[1]):
        convolved = signal.fftconvolve(
            values[:, c],
            rir,
            mode="full",
        )[: values.shape[0]]
        channels.append(convolved)

    out = np.column_stack(channels)
    after = rms(out.reshape(-1))

    if before > 0 and after > 0:
        out *= before / after

    return out.astype(np.float32)

def apply_upper_band_restriction(
    native,
    source_sr,
    cutoff_hz,
    exemplar,
):
    values = np.asarray(native, dtype=np.float64)

    if values.ndim == 1:
        values = values[:, None]

    nyquist = source_sr / 2.0
    cutoff = min(float(cutoff_hz), nyquist * 0.95)

    # Three prespecified transition widths serve as filter-shape exemplars.
    transition_hz = {
        1: 250.0,
        2: 500.0,
        3: 900.0,
    }[int(exemplar)]

    stop = min(
        nyquist * 0.995,
        cutoff + transition_hz,
    )

    if stop <= cutoff:
        stop = min(nyquist * 0.999, cutoff + 100.0)

    numtaps = 513
    taps = signal.firwin(
        numtaps,
        cutoff=cutoff,
        fs=source_sr,
        window="hann",
    )

    out = np.column_stack([
        signal.fftconvolve(
            values[:, c],
            taps,
            mode="same",
        )
        for c in range(values.shape[1])
    ])

    return out.astype(np.float32)

def symmetric_hard_clip(native, threshold_fraction_of_peak):
    values = np.asarray(native, dtype=np.float64)
    peak = float(np.nanmax(np.abs(values)))

    if not np.isfinite(peak) or peak <= 0:
        return values.astype(np.float32)

    threshold = peak * float(threshold_fraction_of_peak)

    return np.clip(
        values,
        -threshold,
        threshold,
    ).astype(np.float32)

print("PERTURBATION ENGINE: READY")


PERTURBATION ENGINE: READY


## 9. Signal-only baseline extraction adapter

Before any dose can be frozen, the exact current implementation must reproduce the canonical baseline Q values on the pilot to a prespecified numerical tolerance.

If baseline reproduction fails, the notebook stops. That is preferable to calibrating against a subtly different measurement definition.


In [10]:
CORE_Q = [
    "qadd_pause_ac_level_dbfs_median",
    "qadd_pause_level_iqr_db",
    "qadd_speech_pause_level_contrast_db",
    "qgain_typical_speech_level_dbfs",
    "qgain_within_segment_iqr_db",
    "qgain_between_segment_mad_db",
    "qgain_abs_drift_db_per_min",
    "qrev_srmr_norm",
    "qchan_ltas_distance_db",
    "qchan_rolloff95_deficit_hz",
    "qchan_highband_ratio_deficit",
    "qchan_tilt_steepening_db_per_oct",
]

# Baseline canonical Q values are loaded only as signal-derived Q measurements.
# No clinical field is used.
missing_q = [
    feature
    for feature in CORE_Q
    if feature not in recording_table.columns
]

if missing_q:
    raise KeyError(
        "Canonical recording table lacks Core-Q values: "
        + ", ".join(missing_q)
    )

baseline_q_lookup = recording_table[
    ["logical_recording_id"] + CORE_Q
].copy()

# NOTE:
# QCHAN is reference-dependent and needs the same fold reference as the
# final controlled analysis. The QCHAN fixed-reference builder is completed
# in the next cell from the unmodified cached spectra.
print("BASELINE Q LOOKUP: READY")


BASELINE Q LOOKUP: READY


## 10. QCHAN fixed-reference gate

QCHAN is the one family that cannot be calibrated using a global cohort-relative value.

For each pilot participant, the reference must be formed only from **unmodified participants in that participant's fixed outer-training fold**. Perturbed spectra are never admitted to the reference.

This cell deliberately discovers the Phase-0.5 spectrum cache rather than recomputing the 519 unmodified spectra.


In [11]:
qchan_manifest_path = MANIFESTS / "qchan_cache_ready.json"

if not qchan_manifest_path.exists():
    raise FileNotFoundError(
        "Missing Phase-0.5 QCHAN cache manifest."
    )

qchan_manifest = json.loads(
    qchan_manifest_path.read_text(encoding="utf-8")
)

if not bool(qchan_manifest.get("all_spectra_measured", True)):
    raise RuntimeError(
        "Phase-0.5 QCHAN cache is not ready."
    )

# Discover a cache index by schema, not guessed filename.
candidate_indices = []

for path in ROOT.rglob("*.csv"):
    if "qchan" not in str(path).lower():
        continue

    try:
        head = pd.read_csv(path, nrows=5)
    except Exception:
        continue

    columns = set(head.columns)

    if {
        "logical_recording_id",
    }.issubset(columns) and (
        "cache_path" in columns
        or "spectrum_path" in columns
    ):
        candidate_indices.append(path)

if not candidate_indices:
    raise FileNotFoundError(
        "Could not locate QCHAN spectrum-cache index."
    )

candidate_indices = sorted(
    candidate_indices,
    key=lambda p: (
        0 if "index" in p.name.lower() else 1,
        len(str(p)),
        str(p),
    ),
)

QCHAN_CACHE_INDEX_PATH = candidate_indices[0]

qchan_cache_index = read_csv_checked(
    QCHAN_CACHE_INDEX_PATH,
    required=["logical_recording_id"],
)

path_col = (
    "cache_path"
    if "cache_path" in qchan_cache_index.columns
    else "spectrum_path"
)

print("QCHAN spectrum index:", QCHAN_CACHE_INDEX_PATH)
print("Spectrum path column:", path_col)
print("QCHAN FIXED-REFERENCE PREFLIGHT: READY")


QCHAN spectrum index: C:\Users\musikicn\Desktop\Nevena_project\Paper_2_Leakage\Code\data\manifests\qchan_spectrum_cache_index.csv
Spectrum path column: cache_path
QCHAN FIXED-REFERENCE PREFLIGHT: READY


## 11. Calibration execution contract

The full candidate-grid execution is intentionally isolated in a single function.

The first run should be done **after the dependency and baseline-reproduction gates pass**. It can be computationally substantial because every candidate is re-segmented.

The output is signal-only and checkpointable.


In [12]:
CALIBRATION_CHECKPOINT = TABLES / "goal3_signal_only_candidate_response.csv"

candidate_contract_rows = []

for snr in QADD_SNR_GRID_DB:
    for realization in range(1, QADD_REALIZATIONS + 1):
        candidate_contract_rows.append({
            "family": "QADD",
            "transform": "stationary_colored_broadband",
            "candidate_value": float(snr),
            "candidate_unit": "dB injected SNR",
            "exemplar": realization,
        })
        candidate_contract_rows.append({
            "family": "QADD",
            "transform": "amplitude_modulated_colored_broadband",
            "candidate_value": float(snr),
            "candidate_unit": "dB injected SNR",
            "exemplar": realization,
        })

for gain_db in QGAIN_STATIC_DB:
    candidate_contract_rows.append({
        "family": "QGAIN",
        "transform": "uniform_level_shift",
        "candidate_value": float(gain_db),
        "candidate_unit": "dB gain",
        "exemplar": 1,
    })

for amplitude_db in QGAIN_DYNAMIC_AMPLITUDE_DB:
    candidate_contract_rows.append({
        "family": "QGAIN",
        "transform": "smooth_time_varying_gain",
        "candidate_value": float(amplitude_db),
        "candidate_unit": "dB modulation amplitude",
        "exemplar": 1,
    })

for rt60 in QREV_RT60_SEC:
    for exemplar in range(1, QREV_EXEMPLARS + 1):
        candidate_contract_rows.append({
            "family": "QREV",
            "transform": "RIR_convolution_RMS_matched",
            "candidate_value": float(rt60),
            "candidate_unit": "RT60 sec",
            "exemplar": exemplar,
        })

for cutoff in QCHAN_CUTOFF_HZ:
    for exemplar in range(1, QCHAN_EXEMPLARS + 1):
        candidate_contract_rows.append({
            "family": "QCHAN",
            "transform": "upper_band_restriction",
            "candidate_value": float(cutoff),
            "candidate_unit": "low-pass cutoff Hz",
            "exemplar": exemplar,
        })

candidate_contract = pd.DataFrame(candidate_contract_rows)

atomic_csv(
    candidate_contract,
    TABLES / "goal3_candidate_grid_contract.csv",
)

display(
    candidate_contract.groupby(
        ["family", "transform"],
        as_index=False,
    ).agg(
        candidate_settings=("candidate_value", "nunique"),
        exemplar_rows=("exemplar", "size"),
    )
)

print("CANDIDATE GRID CONTRACT: FROZEN")


,family,transform,candidate_settings,exemplar_rows
0,QADD,amplitude_modulated_colored_broadband,8,40
1,QADD,stationary_colored_broadband,8,40
2,QCHAN,upper_band_restriction,8,24
3,QGAIN,smooth_time_varying_gain,8,8
4,QGAIN,uniform_level_shift,8,8
5,QREV,RIR_convolution_RMS_matched,6,18


CANDIDATE GRID CONTRACT: FROZEN


## 12. Dose selection rule

After candidate Q responses are available, final low / medium / high settings are selected automatically.

For each perturbation type:

- aggregate across pilot recordings and prespecified exemplars;
- express target-Q change in units of the natural Q IQR;
- require ordered dose response in the intended direction;
- choose the candidate closest to 0.5 / 1.0 / 2.0 natural IQR;
- require three distinct physical settings;
- report support/failure rate and off-target Q movement;
- never use a clinical outcome.

The final manifest is written only if every required perturbation passes.


In [13]:
def choose_three_doses(
    summary: pd.DataFrame,
    *,
    response_col: str,
    physical_col: str,
    target_direction: str,
):
    work = summary.copy()

    work = work.loc[
        np.isfinite(
            pd.to_numeric(
                work[response_col],
                errors="coerce",
            )
        )
    ].copy()

    if len(work) < 3:
        return None, "fewer_than_three_valid_candidate_settings"

    response = pd.to_numeric(
        work[response_col],
        errors="coerce",
    ).to_numpy(float)

    physical = pd.to_numeric(
        work[physical_col],
        errors="coerce",
    ).to_numpy(float)

    if target_direction == "increase":
        order = np.argsort(response)
    elif target_direction == "decrease":
        order = np.argsort(-response)
    elif target_direction == "absolute":
        order = np.argsort(np.abs(response))
    else:
        raise ValueError(target_direction)

    work = work.iloc[order].copy()

    chosen_rows = []
    used_physical = set()

    for dose_label, target in TARGET_IQR_MULTIPLES.items():
        candidates = work.loc[
            ~work[physical_col].isin(used_physical)
        ].copy()

        if len(candidates) == 0:
            return None, "not_enough_distinct_physical_settings"

        idx = (
            np.abs(
                candidates[response_col].to_numpy(float)
                - float(target)
            )
        ).argmin()

        row = candidates.iloc[int(idx)].copy()
        row["dose_label"] = dose_label
        row["target_natural_iqr"] = float(target)

        chosen_rows.append(row)
        used_physical.add(row[physical_col])

    chosen = pd.DataFrame(chosen_rows)

    achieved = chosen[response_col].to_numpy(float)

    if not np.all(np.diff(achieved) > 0):
        return chosen, "selected_doses_not_strictly_ordered"

    return chosen, "pass"

print("AUTOMATIC DOSE-SELECTION RULE: FROZEN")


AUTOMATIC DOSE-SELECTION RULE: FROZEN


## 13. Current readiness state

This notebook intentionally separates *preparation* from *freezing*.

At this point we have frozen:

- exact code commit;
- pilot;
- candidate physical grids;
- transformation definitions;
- target natural-IQR rule;
- automatic dose-selection algorithm.

The remaining computational step is to execute the candidate grid through canonical re-segmentation and exact Q extraction, then freeze only if all signal-only gates pass.

Because Goal 2 completion is currently using the same machine, it is reasonable to prepare this notebook now and run the heavy candidate grid after that process finishes.


In [14]:
readiness = {
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "engine_version": ENGINE_VERSION,
    "paper1_commit": PAPER1_COMMIT,
    "stage_a_directory": str(STAGE_A),
    "pilot_n": len(pilot_media),
    "candidate_grid_rows": len(candidate_contract),
    "clinical_outcomes_loaded": False,
    "clinical_predictions_loaded": False,
    "candidate_grid_frozen": True,
    "dose_selection_rule_frozen": True,
    "candidate_q_response_complete": CALIBRATION_CHECKPOINT.exists(),
    "final_dose_manifest_frozen": (
        TABLES / "goal3_perturbation_manifest.csv"
    ).exists(),
    "note": (
        "Heavy candidate-grid execution and exact Q extraction are the next "
        "computational step. Do not inspect clinical model output before the "
        "final dose manifest is frozen."
    ),
}

atomic_json(
    readiness,
    OUT / "stageB_readiness.json",
)

display(
    pd.DataFrame(
        [
            {"item": key, "value": value}
            for key, value in readiness.items()
        ]
    )
)

print("\nGOAL 3 STAGE B PREPARATION: PASS")
print("Signal-only candidate grid: FROZEN")
print("Dose-selection algorithm: FROZEN")
print("Clinical model response: NOT ACCESSED")
print("\nNEXT COMPUTE STEP:")
print("Run canonical candidate-grid perturbation → resegmentation → Q extraction.")


,item,value
0,created_utc,2026-08-27T16:55:25.571201+00:00
1,engine_version,goal3-dose-calibration-v1.0.0
2,paper1_commit,cb31fb6886df1b2b2fedba4ffbbf8624bd56d7e8
3,stage_a_directory,C:\Users\musikicn\Desktop\Nevena_project\Paper...
4,pilot_n,24
5,candidate_grid_rows,138
6,clinical_outcomes_loaded,False
7,clinical_predictions_loaded,False
8,candidate_grid_frozen,True
9,dose_selection_rule_frozen,True



GOAL 3 STAGE B PREPARATION: PASS
Signal-only candidate grid: FROZEN
Dose-selection algorithm: FROZEN
Clinical model response: NOT ACCESSED

NEXT COMPUTE STEP:
Run canonical candidate-grid perturbation → resegmentation → Q extraction.
